# Compare Models

In this notebook I want to compare models - in particular I want to scrutinize, why almost all metrics of "non_aug_run" and "copy_author_run" are the same, except that in the latter one the Chamfer Distance is significantly smaller. This makes no sense to me, as Command and Argument Loss are almost the same.

Input: 
- Model 1
- Model 2
- sampel id

Output:
- printed list comapring MSE, Cmd.-Loss, Arg.-Loss, CD or Invalid
- Running scores

Then I can run the entire evaluation script and see in real time how it goes

In [1]:
import sys
import os
import random
import torch
import importlib
import numpy as np
sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
sys.path.append("..")
sys.path.append("../code")

from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from dataset import PointCloudEmbeddingSequenceDataset, PointCloudEmbeddingDataset
from models.DeepCAD.cadlib.macro import ALL_COMMANDS, CMD_ARGS_MASK, EOS_IDX, SOL_IDX, EXT_IDX, ARC_IDX
from models.DeepCAD.cadlib.visualize import vec2CADsolid, CADsolid2pc
from models.DeepCAD.utils import read_ply
from scipy.spatial import cKDTree as KDTree



In [23]:
import warnings

# Ignore any warning matching this pattern
warnings.filterwarnings("ignore", message=".*faces have been skipped due to null triangulation")


In [2]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

In [3]:
def load_pointnet(model_path):

    # Get torch model dir
    model_dir = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    config = model_dir['config']
 #   print(f"Loading PointNet++ from {os.path.abspath(model_path)}:")
  #  for key, value in config.items():
   #     print(f"{key}: {value}")
    #print("")

    # Create model
    model = importlib.import_module(config['model_type'])
    if 'architecture' in config:
        if config['architecture'] == 'own':
            classifier = model.get_model(256, normal_channel=False)
        elif config['architecture'] == "copy_author":
            classifier = model.get_model_new(256, normal_channel=False)
        else:
            raise ValueError(f"Invalid architecture '{args.arch}'. Choose either 'own' or 'copy_author'.")
    else:
        classifier = model.get_model(256, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    # Load pre-trained model
    state_dict = model_dir['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    classifier.eval()
    classifier.load_state_dict(state_dict)
    
    return classifier, criterion

In [4]:
def load_deepcad(cfg):
    print("Loading DeepCAD: ")
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

In [5]:
def infer_pointnet(model, criterion, data):
    pc, z_target, vec_target, id = data['pc'], data['z'], data['tgt_vec'], data['id']
    pc = pc.unsqueeze(0)
    z_target = z_target.unsqueeze(0)
    with torch.no_grad():
        pc = pc.transpose(2, 1)
        z_pred, _ = model(pc)
        loss = criterion(z_pred, z_target)
    return z_pred, loss.item()

In [6]:
def infer_deepcad(model, z_pred, data):
    vec_target = data['tgt_vec']
    z_pred = z_pred.unsqueeze(dim = 1)
    with torch.no_grad():
        output = model.decode(z_pred)
        output["tgt_commands"] = vec_target[:, 0].unsqueeze(0)
        output["tgt_args"] = vec_target[:, 1:].unsqueeze(0)
        loss_dict = model.loss_func(output)
        cmd_loss = loss_dict['loss_cmd'].item()
        arg_loss = loss_dict['loss_args'].item()
        vec_pred = model.logits2vec(output)
        return vec_pred, cmd_loss, arg_loss

In [7]:
def calculate_ACC(results):

    TOLERANCE = 3

    # overall accuracy
    avg_cmd_acc = [] # ACC_cmd
    avg_param_acc = [] # ACC_param
    
    # accuracy w.r.t. each command type
    each_cmd_cnt = np.zeros((len(ALL_COMMANDS),))
    each_cmd_acc = np.zeros((len(ALL_COMMANDS),))

    # accuracy w.r.t each parameter
    args_mask = CMD_ARGS_MASK.astype(np.float32)
    N_ARGS = args_mask.shape[1]
    each_param_cnt = np.zeros([*args_mask.shape])
    each_param_acc = np.zeros([*args_mask.shape])

    B = results["tgt_commands"].shape[0]

    for i in range(B): 
        seq_length = list(results["tgt_commands"][i]).index(EOS_IDX)
        out_cmd = results["pred"][i,:seq_length,0]
        gt_cmd = results["tgt_commands"][i, :seq_length].numpy()
    
        out_param = results["pred"][i,:seq_length,1:]
        gt_param = results["tgt_args"][i, :seq_length].numpy()

        cmd_acc = (out_cmd == gt_cmd).astype(np.int32)
        param_acc = []
              
        for j in range(len(gt_cmd)):
            cmd = gt_cmd[j]
            each_cmd_cnt[cmd] += 1
            each_cmd_acc[cmd] += cmd_acc[j]
            if cmd in [SOL_IDX, EOS_IDX]:
                continue
        
            if out_cmd[j] == gt_cmd[j]: # NOTE: only account param acc for correct cmd
                tole_acc = (np.abs(out_param[j] - gt_param[j]) < TOLERANCE).astype(np.int32)
                if cmd == EXT_IDX:
                    tole_acc[-2:] = (out_param[j] == gt_param[j]).astype(np.int32)[-2:]
                elif cmd == ARC_IDX:
                    tole_acc[3] = (out_param[j] == gt_param[j]).astype(np.int32)[3]
                valid_param_acc = tole_acc[args_mask[cmd].astype(bool)].tolist()
                param_acc.extend(valid_param_acc)
                each_param_cnt[cmd, np.arange(N_ARGS)] += 1
                each_param_acc[cmd, np.arange(N_ARGS)] += tole_acc

        if len(param_acc) == 0: 
            param_acc = 0
        else:
            param_acc = np.mean(param_acc)
        
        avg_param_acc.append(param_acc)
        cmd_acc = np.mean(cmd_acc)
        avg_cmd_acc.append(cmd_acc)

    avg_cmd_acc = np.mean(avg_cmd_acc)
    avg_param_acc = np.mean(avg_param_acc)
    
    each_cmd_acc = each_cmd_acc / (each_cmd_cnt + 1e-6)

    # acc of each parameter type
    each_param_acc = each_param_acc * args_mask
    each_param_cnt = each_param_cnt * args_mask
    each_param_acc = each_param_acc / (each_param_cnt + 1e-6)
    return avg_cmd_acc, avg_param_acc

In [48]:
def calculate_CD(vec_pred, gt_pc_path, vec_target, data_id):

    seq_len = vec_target[:,0].tolist().index(EOS_IDX)
    vec_pred = vec_pred.squeeze()[:seq_len]

    try:
        shape = vec2CADsolid(vec_pred) # out vec only contains until target seq length
    except Exception as e:
        return float('nan') # Create CAD failed
    
    try:
        out_pc = CADsolid2pc(shape, 2000, data_id) # 2000 is the number of sampled points
    except Exception as e:
        return float('nan') # Create PC failed

    if np.max(np.abs(out_pc)) > 2: # normalize out-of-bound data
        out_pc = normalize_pc(out_pc)

    gt_pc = read_ply(gt_pc_path)
    sample_idx = random.sample(list(range(gt_pc.shape[0])), 2000)
    gt_pc = gt_pc[sample_idx]

    cd = chamfer_dist(gt_pc, out_pc)
    return cd


In [49]:
def chamfer_dist(gt_points, gen_points, offset=0, scale=1):
    gen_points = gen_points / scale - offset

    # one direction
    gen_points_kd_tree = KDTree(gen_points)
    one_distances, one_vertex_ids = gen_points_kd_tree.query(gt_points)
    gt_to_gen_chamfer = np.mean(np.square(one_distances))

    # other direction
    gt_points_kd_tree = KDTree(gt_points)
    two_distances, two_vertex_ids = gt_points_kd_tree.query(gen_points)
    gen_to_gt_chamfer = np.mean(np.square(two_distances))

    return gt_to_gen_chamfer + gen_to_gt_chamfer

In [50]:
def normalize_pc(points):
    scale = np.max(np.abs(points))
    points = points / scale
    return points

In [79]:
def print_header():
    print("MSE * 10^-3, ACC * 10^-2, CD * 10^-3")
    print(f"{'i':<4} | {'id':<8} | {'mse1':<6} | {'mse2':<6} | {'cmd_loss1':<10} | {'cmd_loss2':<10} | {'arg_loss1':<10} | {'arg_loss2':<10} | {'cmd_acc1':<8} | {'arg_acc1':<8} | {'cmd_acc2':<8} | {'arg_acc2':<8} | {'cd1':<8} | {'cd2':<8} | {'rs_cd1':<8} | {'rs_cd2':<8}")
    print("-" * 175)


def print_line(data, i):
    print(
        f"{data['i'][i]:<4} | {data['id'][i]:<8} | {data['mse1'][i]*1e3:6.2f} | {data['mse2'][i]*1e3:6.2f} | "
        f"{data['cmd_loss1'][i]:10.4f} | {data['cmd_loss2'][i]:10.4f} | {data['arg_loss1'][i]:10.4f} | {data['arg_loss2'][i]:10.4f} | "
        f"{data['cmd_acc1'][i]*1e2:8.2f} | {data['arg_acc1'][i]*1e2:8.2f} | {data['cmd_acc2'][i]*1e2:8.2f} | {data['arg_acc2'][i]*1e2:8.2f} | "
        f"{data['cd1'][i]*1e3:8.2f} | {data['cd2'][i]*1e3:8.2f} | {data['rs_cd1'][i]*1e3:8.2f} | {data['rs_cd2'][i]*1e3:8.2f} | "
    )


In [80]:
pointnet1_path = "experiments/best_4.pth"
pointnet2_path = "experiments/best_5.pth"
cfg = ConfigAE('test', model_path="../data/latent", parse=False)

In [81]:
pointnet1, criterion1 = load_pointnet(pointnet1_path)
pointnet2, criterion2 = load_pointnet(pointnet2_path)
deepcad = load_deepcad(cfg)

Loading DeepCAD: 
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [82]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
metrics_dict = {"i": [],
                "id": [],
                "mse1": [],
                "mse2": [],
                "cmd_loss1": [],
                "cmd_loss2": [],
                "arg_loss1": [],
                "arg_loss2": [],
                "cmd_acc1": [],
                "arg_acc1": [],
                "cmd_acc2": [],
                "arg_acc2": [],
                "cd1": [],
                "cd2": [],
                "rs_cd1": [],
                "rs_cd2": []}
col_width = print_header()

for i, data in enumerate(dataset):
    # Infer PointNets
    z_pred1, mse1 = infer_pointnet(pointnet1, criterion1, data)
    z_pred2, mse2 = infer_pointnet(pointnet2, criterion2, data)

    # Infer DeepCAD
    vec_pred1, cmd_loss1, arg_loss1 = infer_deepcad(deepcad, z_pred1, data)
    vec_pred2, cmd_loss2, arg_loss2 = infer_deepcad(deepcad, z_pred2, data)

    # Calculate Accuracy
    cmd_acc1, arg_acc1 = calculate_ACC({"tgt_commands": data['tgt_vec'].unsqueeze(0)[:,:,0], "tgt_args": data['tgt_vec'].unsqueeze(0)[:,:,1:], "pred": vec_pred1})
    cmd_acc2, arg_acc2 = calculate_ACC({"tgt_commands": data['tgt_vec'].unsqueeze(0)[:,:,0], "tgt_args": data['tgt_vec'].unsqueeze(0)[:,:,1:], "pred": vec_pred2})

    cd1 = calculate_CD(vec_pred1, dataset.get_pc_path(i), data['tgt_vec'], data['id'])
    cd2 = calculate_CD(vec_pred1, dataset.get_pc_path(i), data['tgt_vec'], data['id'])
    

    # Record Metrics
    metrics_dict['i'].append(i)
    metrics_dict['id'].append(data['id'])
    metrics_dict['mse1'].append(mse1)
    metrics_dict['mse2'].append(mse2)
    metrics_dict['cmd_loss1'].append(cmd_loss1)
    metrics_dict['cmd_loss2'].append(cmd_loss2)
    metrics_dict['arg_loss1'].append(arg_loss1)
    metrics_dict['arg_loss2'].append(arg_loss2)
    metrics_dict['cmd_acc1'].append(cmd_acc1)
    metrics_dict['arg_acc1'].append(arg_acc1)
    metrics_dict['cmd_acc2'].append(cmd_acc2)
    metrics_dict['arg_acc2'].append(arg_acc2)
    metrics_dict['cd1'].append(cd1)
    metrics_dict['cd2'].append(cd2)
    metrics_dict['rs_cd1'].append(np.nanmedian(metrics_dict['cd1']))
    metrics_dict['rs_cd2'].append(np.nanmedian(metrics_dict['cd2']))
    
    print_line(metrics_dict, i)
    if i ==2:
        break

MSE * 10^-3, ACC * 10^-2, CD * 10^-3
i    | id       | mse1   | mse2   | cmd_loss1  | cmd_loss2  | arg_loss1  | arg_loss2  | cmd_acc1 | arg_acc1 | cmd_acc2 | arg_acc2 | cd1      | cd2      | rs_cd1   | rs_cd2  
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
0    | 00250456 |  10.35 |  10.37 |     0.0000 |     0.0000 |     0.0000 |     0.0000 |   100.00 |   100.00 |   100.00 |   100.00 |     4.20 |     4.33 |     4.20 |     4.33 | 
1    | 00440420 |  94.21 |  95.58 |    10.9277 |     2.9377 |    21.9601 |    26.1948 |    70.00 |    25.00 |    70.00 |    33.33 |    21.87 |    22.41 |    13.03 |    13.37 | 
2    | 00819758 | 106.68 | 107.43 |     5.3568 |     5.1952 |     9.5430 |    11.2804 |    71.43 |    35.71 |    42.86 |    37.50 |      nan |      nan |    13.03 |    13.37 | 
